# Variational Autoencoder (VAE) and the ELBO — Solution

**Difficulty**: 🟡 Medium

---

### Problem Statement

A **variational autoencoder** (Kingma & Welling, 2014) is a latent-variable generative model trained by maximizing the **evidence lower bound (ELBO)**:

$$\log p_\theta(x) \;\ge\; \underbrace{\mathbb{E}_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]}_{\text{reconstruction}} \;-\; \underbrace{\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,\mathcal N(0, I)\big)}_{\text{regularizer}}$$

The encoder outputs a diagonal Gaussian $q_\phi(z\mid x) = \mathcal N\big(\mu, \operatorname{diag}(\sigma^2)\big)$, the decoder outputs one Bernoulli **logit** per pixel, and the **reparameterization trick** $z = \mu + \sigma \odot \varepsilon,\ \varepsilon \sim \mathcal N(0, I)$ makes the sample a differentiable function of $(\mu, \sigma)$ so the whole thing trains with backprop.

This is not just history: Stable Diffusion, SDXL, SD3 and Flux all run diffusion in the latent space of a KL-regularized VAE, β-VAEs are the standard disentanglement baseline, and "why is the KL analytic / why reparameterize / why are VAE samples blurry" are classic interview questions. You will implement the model and its loss (the negative ELBO) and train it on tiny 8×8 binary "bars" images.

---

### Requirements

1. **`VAE(input_dim, hidden_dim, latent_dim)`** — an `nn.Module` with these methods:
   - `encode(x)` → `(mu, logvar)`, each of shape `(B, latent_dim)`: an MLP trunk (`Linear → ReLU`, at least one hidden layer of width `hidden_dim`) followed by **two separate linear heads**. `logvar` is $\log \sigma^2$.
   - `reparameterize(mu, logvar)` → `z = mu + exp(0.5 * logvar) * eps` with `eps = torch.randn_like(mu)` — a single `randn_like` call and no other random op, so a fixed seed reproduces it. It must sample in training mode (returning `mu` in eval mode is optional).
   - `decode(z)` → **logits** of shape `(B, input_dim)` — no sigmoid at the end.
   - `forward(x)` → `(recon_logits, mu, logvar)`: encode, reparameterize, decode the **sampled** `z`.
   - `sample(n)` → `torch.sigmoid(decode(z))` with `z = torch.randn(n, latent_dim)` drawn from the prior: Bernoulli means in $[0, 1]$, shape `(n, input_dim)`.
2. **`vae_loss(recon_logits, x, mu, logvar, beta=1.0)`** → `(loss, recon, kl)`, three scalar tensors:
   - `recon` = binary cross-entropy **with logits**, **summed over pixels** and **averaged over the batch** — i.e. $-\log p_\theta(x \mid z)$ per image.
   - `kl` $= -\tfrac12 \sum_{j=1}^{L}\big(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\big)$, summed over latent dims and averaged over the batch.
   - `loss = recon + beta * kl` — the negative ELBO per image (a β-VAE when β ≠ 1).
3. Inputs `x` are flattened images of shape `(B, input_dim)` with values in $\{0, 1\}$.

---

### Constraints

- ✅ `F.binary_cross_entropy_with_logits` is allowed (and recommended).
- ✅ Keep the loss differentiable — no `.item()` / `.detach()` inside it.
- ❌ Don't use `torch.distributions` in your solution (no `Normal(...).rsample()`, no `kl_divergence`) — the tests use it as the oracle.
- ❌ Don't apply a sigmoid and then `F.binary_cross_entropy` (or `log(p + eps)`) — it saturates for large logits.

---

<details>
  <summary>💡 Hint</summary>

  - Why predict `logvar` instead of `sigma`? A linear head can output any real number and `exp` keeps the variance positive. The standard deviation is $\sigma = \exp(\tfrac12 \cdot \text{logvar})$.
  - For $q = \mathcal N(\mu, \sigma^2)$ and $p = \mathcal N(0, 1)$, one dimension contributes $\tfrac12(\mu^2 + \sigma^2 - 1 - \log \sigma^2)$ — no sampling needed for the KL.
  - Both terms are *per image* quantities (sum over that image's pixels / latent dims), then you average over images. Averaging over pixels instead silently re-weights the KL by the number of pixels.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence   # used by the tests only

torch.manual_seed(0)

In [ ]:
def make_bars(n, size=8, seed=0):
    """Binary 'bars' images: each image switches on a random subset of full rows OR columns.
    Returns (n, size * size) floats in {0, 1}."""
    g = torch.Generator().manual_seed(seed)
    lines = (torch.rand(n, size, generator=g) < 0.25).float()      # which rows / cols are on
    vertical = (torch.rand(n, 1, 1, generator=g) < 0.5).float()     # columns (1) or rows (0)?
    rows = lines[:, :, None].expand(n, size, size)
    cols = lines[:, None, :].expand(n, size, size)
    return (vertical * cols + (1 - vertical) * rows).reshape(n, size * size)


def show(imgs, ncol=8, size=8):
    """Print up to `ncol` images side by side as text (# = pixel on)."""
    imgs = imgs.reshape(-1, size, size)[:ncol]
    for r in range(size):
        print("  ".join("".join("#" if v > 0.5 else "." for v in img[r].tolist()) for img in imgs))


data = make_bars(2048, seed=0)
print(f"data: {tuple(data.shape)}, fraction of pixels on = {data.mean():.3f}")
show(data)

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)        # two separate heads
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),                 # logits: no sigmoid
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)                         # sigma, not sigma^2
        eps = torch.randn_like(std)                           # the only randomness
        return mu + std * eps                                 # differentiable in mu and logvar

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    @torch.no_grad()
    def sample(self, n):
        z = torch.randn(n, self.latent_dim, device=self.fc_mu.weight.device)
        return torch.sigmoid(self.decode(z))


def vae_loss(recon_logits, x, mu, logvar, beta=1.0):
    B = x.shape[0]
    # -log p(x|z): sum over pixels, mean over the batch. The logits version is stable.
    recon = F.binary_cross_entropy_with_logits(recon_logits, x, reduction="sum") / B
    # analytic KL(N(mu, sigma^2) || N(0, I)): sum over latent dims, mean over the batch
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B
    return recon + beta * kl, recon, kl

In [ ]:
# Test 1: shapes, z = mu + sigma * eps, and pathwise gradients
torch.manual_seed(0)
model = VAE(64, 128, 8)
recon_logits, mu, logvar = model(data[:16])
assert recon_logits.shape == (16, 64), recon_logits.shape
assert mu.shape == (16, 8) and logvar.shape == (16, 8), (mu.shape, logvar.shape)

mu_t = torch.randn(5, 8, requires_grad=True)
logvar_t = torch.randn(5, 8, requires_grad=True)
torch.manual_seed(1)
z = model.reparameterize(mu_t, logvar_t)
torch.manual_seed(1)
eps = torch.randn_like(mu_t)
assert torch.allclose(z, mu_t + torch.exp(0.5 * logvar_t) * eps, atol=1e-6), "z must be mu + sigma * eps"
z.sum().backward()
assert torch.allclose(mu_t.grad, torch.ones_like(mu_t)), "dz/dmu must be 1"
assert torch.allclose(logvar_t.grad, 0.5 * torch.exp(0.5 * logvar_t.detach()) * eps, atol=1e-6), \
    "dz/dlogvar must be 0.5 * sigma * eps"
print("Test 1 passed: shapes, z = mu + sigma * eps, pathwise gradients")

In [ ]:
# Test 2: the loss against torch oracles
g = torch.Generator().manual_seed(0)
logits = 2 * torch.randn(6, 64, generator=g)
xb = (torch.rand(6, 64, generator=g) < 0.4).float()
mu_b, logvar_b = torch.randn(6, 8, generator=g), torch.randn(6, 8, generator=g)

loss, recon, kl = vae_loss(logits, xb, mu_b, logvar_b, beta=0.5)
recon_ref = F.binary_cross_entropy_with_logits(logits, xb, reduction="none").sum(dim=1).mean()
kl_ref = kl_divergence(Normal(mu_b, torch.exp(0.5 * logvar_b)), Normal(0.0, 1.0)).sum(dim=1).mean()
assert torch.allclose(recon, recon_ref, atol=1e-4), f"recon {recon.item():.4f} vs {recon_ref.item():.4f}"
assert torch.allclose(kl, kl_ref, atol=1e-4), f"kl {kl.item():.4f} vs {kl_ref.item():.4f}"
assert torch.allclose(loss, recon_ref + 0.5 * kl_ref, atol=1e-4), "loss must be recon + beta * kl"

# saturated logits: sigmoid(50) == 1.0 in float32, the logits form must still be exact
_, r, _ = vae_loss(torch.tensor([[50.0, -50.0]]), torch.tensor([[0.0, 1.0]]),
                   torch.zeros(1, 2), torch.zeros(1, 2))
assert torch.isfinite(r) and abs(r.item() - 100.0) < 1e-3, f"expected 100 nats, got {r.item()}"
print("Test 2 passed: recon / KL / beta match the oracles and stay exact for large logits")

In [ ]:
# Test 3: train on the bars dataset (a few seconds on CPU)
torch.manual_seed(0)
model = VAE(input_dim=64, hidden_dim=128, latent_dim=8)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
history = []
for step in range(1500):
    xb = data[torch.randint(0, len(data), (128,))]
    recon_logits, mu, logvar = model(xb)
    loss, recon, kl = vae_loss(recon_logits, xb, mu, logvar)
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(loss.item())
    if step % 300 == 0 or step == 1499:
        print(f"step {step:4d}   -ELBO {loss.item():6.2f}   recon {recon.item():6.2f}   kl {kl.item():5.2f}")

first, last = sum(history[:20]) / 20, sum(history[-100:]) / 100
assert last < 0.5 * first, f"the negative ELBO should fall a lot: {first:.1f} -> {last:.1f}"
print(f"Test 3 passed: -ELBO {first:.1f} -> {last:.1f} nats per image")

In [ ]:
# Reconstructions and samples from the prior
model.eval()
with torch.no_grad():
    x8 = data[:8]
    mu8, _ = model.encode(x8)
    print("inputs:");                               show(x8)
    print("\nreconstructions (decoded posterior mean):"); show(torch.sigmoid(model.decode(mu8)))
    torch.manual_seed(0)
    print("\nsamples from the prior (thresholded at 0.5):"); show(model.sample(8))
# Reconstructions are near perfect, but some prior samples mix rows and columns or come out
# partial: those z land in "holes" of the latent space that q(z|x) never covers. That gap
# between the aggregate posterior and N(0, I) is one reason VAE samples look worse than
# reconstructions (see the follow-ups).

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("vae", VAE=VAE, vae_loss=vae_loss)

## Why it works

**ELBO.** For any $q(z \mid x)$, Jensen's inequality gives
$\log p(x) = \log \mathbb{E}_{q}\big[p(x, z)/q(z\mid x)\big] \ge \mathbb{E}_q[\log p(x\mid z)] - \mathrm{KL}(q(z\mid x)\,\|\,p(z))$,
and the gap is exactly $\mathrm{KL}(q(z\mid x)\,\|\,p(z\mid x))$ — so maximizing the ELBO both fits the data and pulls the encoder toward the true posterior.

**Reconstruction term.** With a Bernoulli decoder, $-\log p(x\mid z) = \sum_{\text{pixels}} \mathrm{BCE}(\text{logit}, x)$. We estimate the expectation over $q$ with a single sample of $z$ per image (unbiased), and the logits form $\max(\ell, 0) - \ell x + \log(1 + e^{-|\ell|})$ never takes the log of a rounded-to-zero probability.

**KL term.** For diagonal Gaussians, $\mathrm{KL}\big(\mathcal N(\mu, \sigma^2)\,\|\,\mathcal N(0, 1)\big) = \tfrac12(\mu^2 + \sigma^2 - 1 - \log\sigma^2)$ per dimension, summed over independent dims — exact, no Monte Carlo noise.

**Reparameterization.** Writing $z = \mu + \sigma \odot \varepsilon$ moves the randomness into $\varepsilon$, which does not depend on the parameters. Then $\nabla_\phi \mathbb{E}_\varepsilon[f(z)] = \mathbb{E}_\varepsilon[\nabla_\phi f(\mu + \sigma\varepsilon)]$, with $\partial z/\partial\mu = 1$ and $\partial z/\partial \log\sigma^2 = \tfrac12\sigma\varepsilon$ — the gradients Test 1 checks.

## Complexity

One forward/backward is two small MLPs: $O\big(B\,(D H + H L)\big)$ time and memory for batch $B$, input $D$, hidden $H$, latent $L$. The KL is $O(BL)$; the single-sample estimator adds no extra passes.

## Common bugs

- **Variance vs std:** `mu + exp(logvar) * eps` uses $\sigma^2$ as the std; the right factor is `exp(0.5 * logvar)`.
- **Non-reparameterized sampling:** `Normal(mu, std).sample()` / `torch.normal(mu, std)` cut the gradient to the encoder; only `rsample()` / `mu + std * eps` keep it.
- **Mismatched reductions:** `F.binary_cross_entropy_with_logits(...)` with the default `mean` averages over pixels too, which silently multiplies β by the number of pixels (64 here, 784 for MNIST) and is a common cause of posterior collapse in "my VAE only outputs the mean image" bugs. Same for averaging the KL over latent dims or summing it over the batch.
- **Sigmoid then BCE:** `F.binary_cross_entropy(torch.sigmoid(logits), x)` clamps $\log$ at $-100$, and `log(p + 1e-7)` caps it near 16 nats — both wrong once logits saturate.
- **Decoding `mu` in `forward`:** that trains a plain autoencoder plus a KL penalty on a variable the decoder never sees.
- **Sigmoid inside `decode`** and then BCE-with-logits on top: applies the sigmoid twice.

## Interview follow-ups

1. **Why not just use the score-function (REINFORCE) estimator?** It works for any distribution but has much higher variance because it only uses $f(z)$, not $\nabla_z f$; the pathwise (reparameterized) gradient exploits the decoder's gradient and typically has orders-of-magnitude lower variance. For discrete latents you need REINFORCE, Gumbel-softmax relaxations, or straight-through estimators (as in VQ-VAE).
2. **What is posterior collapse and how do you fight it?** A powerful decoder (e.g. autoregressive) can model $x$ without $z$, so the optimum sets $q(z\mid x) = p(z)$ and KL → 0. Remedies: KL annealing (warm β from 0 to 1), free bits (only penalize KL above a per-dim floor $\lambda$: $\max(\lambda, \mathrm{KL}_j)$), weaker decoders, or δ-VAE-style constraints.
3. **What does β do?** β > 1 (β-VAE) trades reconstruction for a more factorized, "disentangled" latent and a tighter information bottleneck; β < 1 gives sharper reconstructions and a less regular latent. With β ≠ 1 the objective is no longer a bound on $\log p(x)$.
4. **Why are VAE samples blurry?** A factorized Gaussian/Bernoulli likelihood averages over all plausible outputs (mean-seeking), the single Gaussian posterior is too simple, and maximum likelihood under the KL penalty spreads mass between modes. Fixes: perceptual / adversarial losses (VQGAN), hierarchical VAEs (NVAE, VDVAE), or using the VAE only as a compressor.
5. **How is a VAE used in latent diffusion?** The Stable Diffusion autoencoder is a KL-regularized VAE with a tiny KL weight (~1e-6) plus perceptual (LPIPS) and patch-GAN losses, downsampling 8× into 4 (SD 1/2) or 16 (SD3/Flux) latent channels; the diffusion model then works on these latents (scaled by a fixed factor to ~unit variance), and samples are decoded once at the end.
6. **How would you evaluate a trained VAE?** Report the negative ELBO (an upper bound on NLL) and a tighter importance-weighted estimate (IWAE bound with K samples); for images also report bits/dim, and look at the KL per latent dim to see how many dims are active.